# Probabilistic calibration diagnostics

This example builds a calibrated continuous Gamma forecast and demonstrates the three diagnostics exposed by `ProbabilisticCalibrationPlot`:

- the PIT histogram for distributional calibration;
- the within-series PIT autocorrelation;
- the calibration curve for the exceedance event $Y > c$.

In [2]:
import numpy as np
import pandas as pd
import sys
import os
sys.path.append(os.path.abspath("../.."))
from tinyshift.forecasting.probabilistic import PanelPredictiveForecast
from tinyshift.forecasting.probabilistic.distribution import GammaPredictiveDistribution
from tinyshift.plot import ProbabilisticCalibrationPlot

## Synthetic continuous forecast

The observations are sampled from the same row-aligned Gamma distributions supplied to the forecast. Consequently, the resulting PIT values should be approximately uniform.

In [3]:
rng = np.random.default_rng(42)
n_observations = 300
means = rng.uniform(8.0, 18.0, size=n_observations)
shapes = np.full(n_observations, 5.0)
observed = rng.gamma(shape=shapes, scale=means / shapes)

forecast_frame = pd.DataFrame(
    {
        "unique_id": "series-1",
        "ds": pd.date_range("2025-01-01", periods=n_observations, freq="D"),
        "lambda_t": means,
        "shape": shapes,
    }
)
forecast = PanelPredictiveForecast(
    frame=forecast_frame,
    distribution=GammaPredictiveDistribution(means, shapes),
    model="gamma-demo",
    id_col="unique_id",
    time_col="ds",
)
evaluation = forecast_frame[["unique_id", "ds"]].assign(y=observed)
evaluation.head()

,unique_id,ds,y
0,series-1,2025-01-01,12.791641
1,series-1,2025-01-02,10.265900
2,series-1,2025-01-03,28.600340
3,series-1,2025-01-04,12.498481
4,series-1,2025-01-05,9.542188


## Individual diagnostics

In [4]:
diagnostics = ProbabilisticCalibrationPlot(forecast, evaluation)
diagnostics.pit_histogram(n_bins=10)

In [5]:
diagnostics.pit_acf(max_lag=20)

In [13]:
diagnostics.calibration_curve(threshold=15.0, n_bins=10)

## Combined summary

The summary places all three diagnostics in a single interactive figure.

In [7]:
diagnostics.summary(threshold=15.0, n_bins=10, max_lag=20)